# Домашняя работа 3. Оптимизация и регуляризация изнутри

**Курс «Машинное обучение», 4 курс**

| | |
|---|---|
| К лабораторной | занятие 3 — Градиентный спуск, обусловленность и регуляризация |
| Опора | материал семинара 3 и лекций до него |
| Ожидаемое время | 3–4 часа |
| Данные | **ваша индивидуальная таблица** (по ФИО) |

На занятии мы вызывали `Ridge` и `Lasso` и пользовались готовым градиентным спуском. Дома напишем оптимизацию сами: стохастический спуск с расписаниями шага, LASSO покоординатным спуском (и добьёмся совпадения со `sklearn` до $10^{-10}$), а в конце увидим, что регуляризация — не трюк, а априорное предположение о весах.\n\nКак и в домашней работе 2: контролируемая синтетика нужна там, где требуется заранее известный ответ (сходимость SGD, истинные веса априорного распределения), а выводы проверяются на **вашей** матрице «объекты–признаки».

> **Чем это отличается от занятия.** На семинаре данные были учебные и общие —
> так удобно разбирать. Дома данные ваши: таблица порождается по ФИО, и ни у
> кого в группе она не повторяется. Приёмы те же, числа другие — поэтому
> отвечать придётся за свои числа, а не за преподавательские.


## Как устроена работа

Работа делится на две части, и делятся они по назначению, а не по сложности.

**Обязательная часть — допуск.** Без неё работа не принимается: это тот минимум,
без которого занятие считается неусвоенным. Здесь всегда есть хотя бы одна
реализация «с нуля», сверенная с `scikit-learn` численно.

**Часть на оценку** (помечена значком ★). Она не нужна для допуска — но балл
за работу выставляется именно по ней, и каждый выполненный пункт идёт в зачёт
отдельно. Браться стоит даже за один пункт: это лучше, чем не браться вовсе.

## Что нужно сдать

Заполненный ноутбук, в котором:

1. выполнены все ячейки с `# TODO` в обязательной части, код исполняется сверху
   вниз без ошибок в свежем ядре (Kernel → Restart & Run All);
2. под каждым заданием заполнена ячейка **Вывод** — своими словами,
   со ссылкой на полученные числа;
3. графики подписаны: заголовок, оси, легенда;
4. в ячейке варианта вписано ваше ФИО.

> Списывание видно сразу: у каждого студента свой датасет и свой набор методов.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from labdata import load_personal

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

Регистр, лишние пробелы и написание «ё»/«е» роли не играют. Если ФИО вписано неверно, вариант будет чужим — проверьте вывод ячейки.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=3)
describe_variant(variant)

In [ ]:
# Ваша выборка (эталонная предобработка занятия 1)
data = load_personal(variant)
X_tr, X_te = data["X_train"], data["X_test"]
y_tr, y_te = data["y_train"], data["y_test"]
feat = np.asarray(data["feature_names"])

print(f"{data['domain']}: обучающая {X_tr.shape}, контрольная {X_te.shape}")
print(f"cond(X^T X) = {np.linalg.cond(X_tr.T @ X_tr):.3e}")

In [ ]:
# Контролируемая выборка для задач 1 и 3: истинные веса известны
n = 200
x1 = rng.normal(0, 1, n)
X_gd = np.column_stack([x1, 0.6 * x1 + 0.8 * rng.normal(0, 1, n)])
y_gd = X_gd @ np.array([2.0, -1.0]) + rng.normal(0, 0.5, n)
theta_star = np.linalg.lstsq(X_gd, y_gd, rcond=None)[0]
print(f"theta* (эталон) = {np.round(theta_star, 4)}")

---
# Задача 1. Стохастический градиентный спуск

Определение 2.5 использует **усреднённый** функционал
$Q(\theta)=\frac1\ell\sum_i Q_i(\theta)$, где
$Q_i = (\langle\theta,x_i\rangle - y_i)^2$. Тогда
$\nabla Q_i = 2x_i(\langle\theta,x_i\rangle - y_i)$, а для мини-пакета градиент
усредняется по объектам пакета.

Ключевой факт из лекции: при **постоянном** шаге SGD не сходится в точку, а
колеблется вокруг $\theta^*$. Для сходимости нужен убывающий шаг, удовлетворяющий
условию Роббинса–Монро: $\sum_t\eta_t=\infty$, $\sum_t\eta_t^2<\infty$.

In [ ]:
def sgd(X, y, eta0, n_epochs=40, batch_size=1, schedule="const", generator=None):
    """SGD для УСРЕДНЁННОГО Q. Возвращает (theta, история Q по эпохам).

    На каждой эпохе перемешать объекты и идти пакетами размера batch_size.
    Градиент по пакету: 2 X_b^T (X_b theta - y_b) / |B|.
    schedule: 'const' -> eta0; 'sqrt' -> eta0/sqrt(t); 'inverse' -> eta0/t,
    где t -- номер ШАГА (не эпохи).
    """
    raise NotImplementedError

### Задание 1.1. Размер пакета и расписание шага

In [ ]:
lam_avg = np.linalg.eigvalsh(X_gd.T @ X_gd / len(X_gd)).max()
eta0 = 0.5 / lam_avg
Q_star = np.mean((X_gd @ theta_star - y_gd) ** 2)

# TODO: (а) на одном графике сравните batch_size = 1, 8, 32 и полный градиент
#           при постоянном шаге (по оси Q откладывайте Q - Q*, масштаб лог.);
#       (б) на втором -- три расписания шага при batch_size = 1.

In [ ]:
# TODO: проверьте условие Роббинса--Монро численно: для расписаний
#       eta0/sqrt(t) и eta0/t посчитайте суммы eta_t и eta_t^2 на 100000 шагах.

> **Вывод.** Почему при постоянном шаге SGD не сходится, а полный градиент — сходится? Как высота «полки» зависит от размера пакета? Какое расписание удовлетворяет условию Роббинса–Монро?
>
> *(ваш ответ здесь)*

---
# Задача 2. LASSO покоординатным спуском

Явной формулы для LASSO нет, но при фиксированных остальных координатах задача
по одной координате решается точно. Обозначим
$r^{(-j)} = y - \sum_{k\ne j}x_k\theta_k$; минимизируется
$\|r^{(-j)} - x_j\theta_j\|^2 + \lambda|\theta_j|$, и из условия на субградиент

$$
\theta_j = \frac{S\bigl(\langle x_j, r^{(-j)}\rangle,\ \lambda/2\bigr)}{\|x_j\|^2},
\qquad
S(z,\gamma) = \mathrm{sign}(z)\max(|z| - \gamma, 0).
$$

Функция $S$ — «мягкий порог»: она **обнуляет** координату, если её вклад меньше
порога. Отсюда и разреженность, которую мы видели на занятии.

> **Напоминание — субградиент.** $|\theta_j|$ не дифференцируема в нуле, поэтому условие «производная равна
> нулю» неприменимо. Для выпуклой функции его заменяет **субградиент**: любой
> вектор $g$, для которого $f(z)\ge f(\theta) + \langle g, z-\theta\rangle$
> при всех $z$ — то есть любая опорная прямая, лежащая под графиком. У гладкой
> функции субградиент один и равен градиенту, а у $|\theta|$ в нуле это **весь
> отрезок** $[-1, 1]$. Условие минимума выпуклой функции: $0$ принадлежит
> множеству субградиентов. Именно из-за «люфта» $[-1,1]$ в нуле у LASSO целый
> диапазон значений $\langle x_j, r\rangle$ даёт ровно $\theta_j = 0$ — так и
> получается разреженность.

In [ ]:
def soft_threshold(z, gamma):
    """S(z, gamma) = sign(z) * max(|z| - gamma, 0)."""
    raise NotImplementedError


def lasso_cd(X, y, lam, n_iter=300, tol=1e-10):
    """Покоординатный спуск. Первый столбец X -- единицы, он НЕ штрафуется.

    Схема: theta = 0; повторять до сходимости:
      для каждой координаты j пересчитать theta_j по формуле выше,
      поддерживая вектор невязки resid = y - X theta.
    """
    raise NotImplementedError

### Задание 2.1. Сверка со `sklearn`

Внимание на нормировку: `sklearn` минимизирует
$\frac{1}{2\ell}\|y - Xw\|^2 + \alpha\|w\|_1$, а наш функционал —
$\|X\theta - y\|^2 + \lambda\|\theta\|_1$. Выразите $\alpha$ через $\lambda$
и добейтесь совпадения до $10^{-10}$.

In [ ]:
from sklearn.linear_model import Lasso

n_obj = 120
X_sp = rng.normal(size=(n_obj, 12))
theta_sparse = np.zeros(12); theta_sparse[[0, 3, 7]] = [3.0, -2.0, 1.5]
y_sp = X_sp @ theta_sparse + rng.normal(0, 0.3, n_obj)
X_sp1 = np.column_stack([np.ones(n_obj), X_sp])

# TODO: для lambda = 1, 10, 60 сравните lasso_cd со sklearn.linear_model.Lasso.
#       Подберите alpha так, чтобы функционалы совпали (см. текст выше),
#       и выведите расхождение и число ненулевых коэффициентов у обоих.

### Задание 2.2. Регуляризационный путь на ваших данных

Теперь примените свой `lasso_cd` к **вашей** матрице. Постройте
регуляризационный путь: как меняются коэффициенты при росте $\lambda$ — и
назовите признаки, которые держатся дольше всех.

In [ ]:
# TODO: 1) добавьте к X_tr столбец единиц;
#       2) постройте сетку lambda от lam_max = 2*max|X^T (y - mean y)| вниз
#          на четыре порядка (np.logspace, 40 точек);
#       3) для каждой lambda вызовите свой lasso_cd и соберите путь;
#       4) нарисуйте коэффициенты против lambda в логарифмической шкале
#          (ось lambda -- по убыванию);
#       5) назовите 5 признаков, которые остаются ненулевыми дольше всех.

> **Вывод.** Совпало ли с точностью $10^{-10}$? Сколько ненулевых координат осталось при $\lambda = 60$ и совпало ли это с истинным числом полезных признаков?
>
> *(ваш ответ здесь)*

---

> ### ★ Дальше — часть на оценку
>
> Обязательная часть закончилась: если вы дошли досюда и всё работает, работа
> будет принята. Дальше идут задания, по которым выставляется балл. Каждый
> пункт засчитывается отдельно, поэтому имеет смысл сделать хотя бы один.

# Задача 3★. Регуляризация как априорное знание

Утверждение 2.13: при гауссовском шуме $\varepsilon\sim\mathcal N(0,\sigma^2)$
и гауссовском априорном распределении весов $\theta_j\sim\mathcal N(0,\tau^2)$
оценка апостериорного максимума совпадает с гребневой регрессией,
$\lambda = \sigma^2/\tau^2$. Для лапласовского априорного
$p(\theta_j)\propto e^{-|\theta_j|/b}$ получается LASSO с $\lambda = 2\sigma^2/b$.

> **Напоминание — априорное распределение и MAP.** В байесовском подходе параметр $\theta$ считается случайным. **Априорное**
> распределение $p(\theta)$ — то, что мы думаем о нём *до* данных («веса скорее
> малы, чем велики»). По формуле Байеса
> $p(\theta\mid X) \propto p(X\mid\theta)\,p(\theta)$ получается
> **апостериорное** — то, что мы думаем *после*. Оценка **MAP** (maximum a
> posteriori) — точка его максимума:
>
> $$
> \theta_{\mathrm{MAP}} = \arg\max_\theta\bigl[\ln p(X\mid\theta) + \ln p(\theta)\bigr].
> $$
>
> Первое слагаемое — то самое правдоподобие из занятия 2 (при $p(\theta)\equiv$
> const получается обычный ММП), второе — штраф. Отсюда и смысл утверждения 2.13:
> регуляризация не «математический трюк», а ровно запись априорного знания.
> Гауссовское априорное даёт $\sum\theta_j^2$, лапласовское — $\sum|\theta_j|$.

In [ ]:
from scipy import optimize


def ridge_fit(X, y, lam):
    """theta*_lambda по теореме 2.10; первый столбец X -- единицы, не штрафуется."""
    raise NotImplementedError


sigma, tau, b_lap = 0.3, 0.5, 0.4

# TODO: 1) напишите -ln p(theta | X, y) для гауссовского и лапласовского
#          априорных распределений (theta[0] не штрафуется);
#       2) минимизируйте обе численно (для лапласовской -- method="Powell");
#       3) сравните с ridge_fit(sigma^2/tau^2) и lasso_cd(2 sigma^2/b).

In [ ]:
# TODO: нарисуйте обе априорные плотности на одном графике.

> **Вывод.** Совпали ли MAP-оценки с формулами? Какая плотность сильнее «настаивает» на нулевом весе и как это объясняет разреженность LASSO?
>
> *(ваш ответ здесь)*

## Итоги домашней работы

Кратко ответьте на вопросы:

1. Вы взяли шаг $\eta = 0.9/\lambda_{\max}$, и спуск всё равно разошёлся. Назовите две возможные причины.
2. Почему покоординатный спуск удобен именно для LASSO и плохо подходит, например, для задачи с ограничением $\sum_j\theta_j = 1$?

---

Проверьте перед сдачей: Kernel → Restart & Run All проходит без ошибок,
все ячейки **Вывод** заполнены, графики подписаны.